# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Print basic dataset info
metadata = dataset.metadata
print(f"{metadata.name} (Identifier: {metadata.identifier})\n{metadata.description}\n\nPublished: {metadata.datePublished}\nLicense: {metadata.license}\n")

## 2. Data Overview

Review available record sets, their fields and unique `@id`s.

In [ ]:
# List available record sets and their unique @ids
print("Available record sets in the dataset:")
for record_set in metadata.record_sets:
    print(f"  - Name: {record_set.name}; @id: {record_set.id}")

# Print the fields and columns for each record set
for record_set in metadata.record_sets:
    print(f"\n▶️ Record set: {record_set.name} (@id: {record_set.id})")
    if record_set.fields:
        print("Fields:")
        for f in record_set.fields:
            print(f"  - Name: {f.name}; @id: {f.id}; dataType: {f.data_type if hasattr(f, 'data_type') else 'N/A'};")
    if hasattr(record_set, "columns") and record_set.columns:
        print("Columns:")
        for c in record_set.columns:
            print(f"  - Name: {c.name}; @id: {c.id};")

## 3. Data Extraction

Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# In this dataset, we extract all available data tables.

record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    except Exception as e:
        print(f"Could not convert {record_set_id} to DataFrame: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalization, and grouping, referencing all fields and columns by their `@id`.

Let's select a main data record set for demonstration. You may update the `main_record_set_id` if exploring a different record set.

In [ ]:
# Suppose the main proteins data table's @id is 'https://api.app.sen.science/frontiers/7154140/ev_proteins', update as available.

# Find largest table for demonstration
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1]) if dataframes else None
main_df = dataframes[main_record_set_id] if main_record_set_id else None
print(f"Analyzing main record set: {main_record_set_id}")

# List numeric fields (columns likely to have numeric data)
numeric_fields = []
if main_df is not None:
    for col in main_df.columns:
        try:
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_fields.append(col)
        except Exception:
            pass
    print(f"Possible numeric fields (@id): {numeric_fields}")

# Attempt EDA on first numeric field
if numeric_fields:
    numeric_field = numeric_fields[0]  # For demo, select the first one
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].dtype != object else 0
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f})\n")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' field\n")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by a likely categorical field
    candidate_groups = [c for c in main_df.columns if c not in numeric_fields]
    group_field = candidate_groups[0] if candidate_groups else None
    if group_field is not None:
        
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of '{numeric_field}' by '{group_field}'\n")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All field and record set references should use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_fields:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_fields[0]], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_fields[0]} (by @id)")
    plt.xlabel(numeric_fields[0])
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if two numeric fields are available
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(
            data=main_df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs. {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("Not enough numeric data to visualize.")

## 6. Conclusion

This notebook showcased metadata exploration, schema inspection, and example record/table loading for the FAIR² dataset using the `mlcroissant` library.

- Dataset schema and structure are discoverable via the Croissant standard and referenced by their persistent `@id`s.
- You can use the record set and field `@id` to access or manipulate specific tables and attributes.
- For in-depth analyses, refer to the data dictionary and consider field definitions and descriptions to guide your workflows.

Explore further by adapting the EDA and visualization sections to match questions relevant to your research or data science needs.